# LLMDTA_FNet -- Component Ablation (chay thu / smoke test) -- Davis/Warm (Kaggle Notebook)

Notebook nay chay THU (trial run) script `code/ablation_fft.py` -- ablation tung thanh phan kien truc cua model FNet (model dung FourierMixing/FFT thay CrossAttention): Encoder, chinh co che FFT mixing, Self-Attention Pooling, Residual Fusion, MoE, va bien the doi mixing sang CrossAttention.

**8 bien the duoc ablation:**
- `full` -- Model FFT day du (baseline)
- `wo_encoder` -- Linear Encoder thay 1D-CNN (giu FFT)
- `wo_fft_mixing` -- Bo han FFT (Identity passthrough)
- `attn_mixing` -- FFT -> CrossAttention (so sanh mixing)
- `wo_self_attn_pool` -- Mean Pooling thay Self-Attention Pooling (giu FFT)
- `wo_residual` -- Bo residual fusion (giu FFT)
- `wo_moe` -- 1 predictor don thay MoE (giu FFT)
- `linear_only` -- Chi Linear predictor (khong Encoder/Mixing/MoE)

**Muc dich cua ban "chay thu" nay**: xac nhan script chay dung, khong loi, tren toan bo 8 bien the, VOI SO EPOCH RAT NHO (mac dinh 5 epoch/bien the, patience 3) de nhanh co ket qua so bo. Ket qua so bo nay KHONG dai dien cho hieu nang thuc su cua model (can it nhat epochs~100, patience~20, va chay ca 5 fold moi co the ket luan).

Sau khi chay thu thanh cong (khong loi, thay bang ket qua o cuoi notebook), tang `TRIAL_EPOCHS` / `TRIAL_PATIENCE`, bat `--all_folds`, de chay ablation day du (co the mat vai gio tuy GPU va so bien the).

## Truoc khi chay (bat buoc)

1. **Bat GPU**: panel ben phai -> *Settings* -> *Accelerator* -> chon **GPU T4 x2**. KHONG chon P100: nhieu ban PyTorch gan day (>= 2.1) da bo ho tro kernel cho kien truc Pascal (P100, sm_60), gay loi `CUDA error: no kernel image is available for execution on the device` ngay khi train (du P100 ve ly thuyet co bang thong bo nho tot hon cho FFT, no chi co loi neu ban torch hien tai con ho tro no).
2. **Bat Internet**: *Settings* -> *Internet* -> **On** (can de `pip install` va `git clone`).
3. **Add Data**: nut **+ Add Data** (goc phai) -> tim `llmdta` (chu so huu `christang0002`) -> **Add**.

Sau khi lam du 3 buoc tren, chon **Run All**. Cell ngay sau `nvidia-smi` se tu dong kiem tra xem GPU/ban PyTorch hien tai co tuong thich khong -- neu KHONG, no se bao loi ro rang ngay lap tuc thay vi de ban cho het 8 bien the roi moi biet la loi GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('Torch build CUDA:', torch.version.cuda)
print('torch.cuda.is_available():', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        'torch.cuda.is_available() = False -- kiem tra lai Settings -> Accelerator '
        'da bat GPU chua (T4 x2 hoac P100).'
    )

device_name = torch.cuda.get_device_name(0)
cap_major, cap_minor = torch.cuda.get_device_capability(0)
needed_arch = f'sm_{cap_major}{cap_minor}'
arch_list = torch.cuda.get_arch_list()

print('GPU:', device_name, '->', needed_arch)
print('Kernel co san trong ban torch nay cho:', arch_list)

if not any(needed_arch in a for a in arch_list):
    raise RuntimeError(
        f"\nGPU hien tai ({device_name}, {needed_arch}) KHONG duoc ban PyTorch {torch.__version__} nay ho tro "
        f"(chi co kernel bien dich san cho: {arch_list}).\n"
        "Day chinh la nguyen nhan loi 'CUDA error: no kernel image is available for execution on the device' "
        "se xay ra khi train (thay vi cho no chay het 40 luot roi moi bao loi).\n\n"
        "=> CACH SUA: vao Settings -> Accelerator, doi sang GPU T4 x2 (kien truc Turing, sm_75) roi Run lai "
        "tu dau. Nhieu ban PyTorch gan day (>= 2.1) da BO ho tro kernel cho P100 (kien truc Pascal, sm_60) "
        "trong wheel mac dinh, du P100 van ve mat ly thuyet co bang thong bo nho tot hon cho FFT."
    )

print('\nOK: GPU nay duoc ban PyTorch hien tai ho tro day du, an toan de tiep tuc.')

In [ ]:
# Luu y: KHONG pin gensim==4.3.1 (khong co wheel dung san cho Python tren Kaggle
# -> pip phai build tu source va thuong loi). De pip tu chon ban gensim moi nhat.
!pip install -q rdkit gensim mol2vec wandb scikit-learn scipy tqdm
print('Da cai xong dependencies.')

In [ ]:
import os

REPO_URL = 'https://github.com/glucose20org/Temp.git'
REPO_BRANCH = 'quantum'
REPO_ROOT = '/kaggle/working/Temp'

if not os.path.exists(REPO_ROOT):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}
else:
    print('Repo da ton tai, dang pull ban moi nhat...')
    !cd {REPO_ROOT} && git pull origin {REPO_BRANCH}

os.chdir(REPO_ROOT)
print('Working dir:', os.getcwd())

assert os.path.exists('code/ablation_fft.py'), 'Khong tim thay code/ablation_fft.py -- kiem tra lai REPO_BRANCH.'
print('Da tim thay code/ablation_fft.py')

## Tim du lieu pretrain embedding tu dataset da Add Data

Cell duoi tu dong quet toan bo `/kaggle/input/**` de tim 2 file `*_drug_pretrain.pkl` va `*_esm_pretrain.pkl` cho dataset `davis`. Neu ten file trong dataset Kaggle khac pattern doan duoc, notebook se in ra cay thu muc `/kaggle/input` de ban tu xac dinh duong dan, roi gan thu cong vao `MANUAL_DRUG_PKL` / `MANUAL_PROT_PKL` o cell ke tiep.

In [ ]:
import glob, shutil, tarfile

DATASET = 'davis'         # davis | kiba | metz
RUNNING_SET = 'warm'      # warm | novel-drug | novel-prot | novel-pair

# Fold-data (train/valid/test csv) da co san trong repo duoi dang .tar.gz -> chi can giai nen
fold_root = os.path.join(REPO_ROOT, 'data', 'dta-5fold-dataset')
tar_path = os.path.join(fold_root, f'{DATASET}.tar.gz')
extracted_path = os.path.join(fold_root, DATASET)
if not os.path.exists(extracted_path) and os.path.exists(tar_path):
    print(f'Giai nen {tar_path} ...')
    with tarfile.open(tar_path) as tf:
        try:
            tf.extractall(fold_root, filter='data')
        except TypeError:
            tf.extractall(fold_root)  # Python < 3.12 khong co tham so filter
print('Fold-data san sang tai:', extracted_path)

# Ghi de thu cong neu can (de trong '' de dung auto-detect)
MANUAL_DRUG_PKL = ''
MANUAL_PROT_PKL = ''

KAGGLE_INPUT_ROOT = '/kaggle/input'

def find_best_match(patterns, search_root):
    candidates = []
    for pat in patterns:
        candidates += glob.glob(os.path.join(search_root, '**', pat), recursive=True)
    return sorted(set(candidates))

target_dir = os.path.join(REPO_ROOT, 'data', DATASET)
os.makedirs(target_dir, exist_ok=True)
target_drug = os.path.join(target_dir, f'{DATASET}_drug_pretrain.pkl')
target_prot = os.path.join(target_dir, f'{DATASET}_esm_pretrain.pkl')

if MANUAL_DRUG_PKL:
    shutil.copy(MANUAL_DRUG_PKL, target_drug)
    print('Da copy (thu cong) drug pretrain ->', target_drug)
elif not os.path.exists(target_drug):
    drug_candidates = find_best_match([f'*{DATASET}*drug_pretrain*.pkl', f'*{DATASET}*mol2vec*.pkl'], KAGGLE_INPUT_ROOT)
    print('Ung vien drug pretrain:', drug_candidates)
    if drug_candidates:
        shutil.copy(drug_candidates[0], target_drug)
        print('Da copy drug pretrain ->', target_drug)

if MANUAL_PROT_PKL:
    shutil.copy(MANUAL_PROT_PKL, target_prot)
    print('Da copy (thu cong) prot pretrain ->', target_prot)
elif not os.path.exists(target_prot):
    prot_candidates = find_best_match([f'*{DATASET}*esm_pretrain*.pkl', f'*{DATASET}*esm*.pkl'], KAGGLE_INPUT_ROOT)
    print('Ung vien prot pretrain:', prot_candidates)
    if prot_candidates:
        shutil.copy(prot_candidates[0], target_prot)
        print('Da copy prot pretrain ->', target_prot)

if not os.path.exists(target_drug) or not os.path.exists(target_prot):
    print('\nKhong tu dong tim thay du file pretrain. Cay thu muc /kaggle/input hien co:')
    for root, dirs, fs in os.walk(KAGGLE_INPUT_ROOT):
        depth = root.replace(KAGGLE_INPUT_ROOT, '').count(os.sep)
        if depth > 3:
            continue
        print('  ' * depth + os.path.basename(root) + '/')
        for f in fs[:15]:
            print('  ' * (depth + 1) + f)
    print('\nHay kiem tra duong dan chinh xac roi gan vao MANUAL_DRUG_PKL / MANUAL_PROT_PKL o tren va chay lai cell nay.')
    print('Neu chua Add Data: bam "+ Add Data" -> tim "llmdta" (christang0002/llmdta) -> Add.')
else:
    print('\nDa co du 2 file pretrain embedding cho', DATASET)

## Chay thu (trial run) toan bo 8 bien the ablation

Cell duoi goi truc tiep `code/ablation_fft.py` (script da duoc push len GitHub o branch `quantum`) voi so epoch RAT NHO de kiem tra nhanh: khong loi, moi bien the train/eval duoc, va co bang ket qua o cuoi.

- `TRIAL_EPOCHS = 5`, `TRIAL_PATIENCE = 3`: chi de smoke-test, khong phai con so dung de bao cao ket qua.
- Chay 1 fold (`FOLD = 0`) thay vi `--all_folds` de nhanh hon.
- Neu muon test nhanh hon nua, sua `--variants` de chi chay vai bien the (vi du `full,wo_fft_mixing,attn_mixing`).

In [ ]:
DATASET = 'davis'
RUNNING_SET = 'warm'
FOLD = 0

TRIAL_EPOCHS = 5      # CHI DE CHAY THU -- tang len ~100 khi chay that
TRIAL_PATIENCE = 3    # CHI DE CHAY THU -- tang len ~20 khi chay that
TRIAL_BATCH_SIZE = 64
OUTPUT_DIR = '/kaggle/working/ablation_trial_results'

os.makedirs(OUTPUT_DIR, exist_ok=True)

!python code/ablation_fft.py \
    --dataset {DATASET} --running_set {RUNNING_SET} --fold {FOLD} \
    --epochs {TRIAL_EPOCHS} --patience {TRIAL_PATIENCE} --batch_size {TRIAL_BATCH_SIZE} \
    --output_dir {OUTPUT_DIR}

## Doc ket qua chay thu

Neu cell tren chay xong khong bao loi, ket qua (CSV + JSON) da duoc luu vao `OUTPUT_DIR`. Cell duoi doc file CSV moi nhat va hien bang so sanh CI/MSE giua 8 bien the.

In [ ]:
import glob
import pandas as pd

# Loai file *_partial.csv (checkpoint tam thoi, cap nhat sau MOI bien the) khoi ket
# qua "hoan tat" -- chi fallback ve no neu chua co file cuoi cung nao ca (vi du bi
# huy giua chung truoc khi ablation_fft.py kip ghi file tong ket cho fold nay).
all_candidates = glob.glob(os.path.join(OUTPUT_DIR, f'ablation_fnet_{DATASET}_{RUNNING_SET}_fold{FOLD}_*.csv'))
csv_files = sorted(f for f in all_candidates if not f.endswith('_partial.csv'))

if csv_files:
    result_csv = csv_files[-1]
    print('Doc ket qua (da hoan tat fold nay) tu:', result_csv)
else:
    partial_files = sorted(f for f in all_candidates if f.endswith('_partial.csv'))
    assert partial_files, ('Khong tim thay file ket qua nao (ca final lan partial) -- '
                            'kiem tra lai cell chay ablation o tren co bao loi khong.')
    result_csv = partial_files[-1]
    print('Chua co file ket qua hoan tat cho fold nay (co the dang chay do hoac bi huy '
          'giua chung) -- tam doc file checkpoint:', result_csv)

df = pd.read_csv(result_csv)

n_failed = int((df['status'] == 'failed').sum()) if 'status' in df.columns else 0
if n_failed:
    print(f'\nCANH BAO: {n_failed}/{len(df)} bien the bi LOI trong lan chay nay:')
    for _, row in df[df['status'] == 'failed'].iterrows():
        print(f"  - {row['variant']}: {row.get('error', '(khong ro loi)')}")

df_ok = df[df['status'] == 'success'] if 'status' in df.columns else df
assert not df_ok.empty, ('Tat ca bien the da chay deu bi loi -- xem thong bao loi chi tiet o tren '
                          '(va log train/eval phia truoc cell nay) de biet nguyen nhan that su.')

df_sorted = df_ok.sort_values('ci', ascending=False).reset_index(drop=True)
df_sorted[['variant', 'description', 'ci', 'mse', 'r2', 'best_epoch', 'training_time']]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.bar(df_sorted['variant'], df_sorted['ci'], color='#4C72B0')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Test CI')
plt.title(f'FFT Component Ablation (TRIAL, {TRIAL_EPOCHS} epoch/bien the) - {DATASET}-{RUNNING_SET} fold {FOLD}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'trial_ci_comparison.png'), dpi=150)
plt.show()

## Ket qua & buoc tiep theo

Day chi la **chay thu** (1 fold, 5 epoch/bien the) de xac nhan `ablation_fft.py` hoat dong dung tren Kaggle -- CI/MSE o day chi mang tinh minh hoa, chua the dung de ket luan thanh phan nao quan trong.

**De chay ablation THAT (dung de bao cao ket qua):**
1. Tang `TRIAL_EPOCHS` len ~100 va `TRIAL_PATIENCE` len ~20.
2. Them `--all_folds` vao lenh `!python code/ablation_fft.py ...` de chay ca 5 fold va lay mean +/- std.
3. Luu y thoi gian chay: 8 bien the x 5 fold x ~100 epoch se mat kha lau -- nen chay tren session GPU rieng (giong cach 3 notebook `Kaggle_*_Davis_Warm.ipynb` khac tach rieng tung model de tranh vuot quota).
4. Sau khi chay xong, vao tab **Output** cua Kaggle de tai `ablation_fnet_*.csv` / `*.json` / `*_AGGREGATED_*.csv` ve.

## Chay THAT (full ablation, dung de bao cao ket qua) -- chay tren GPU T4 x2

Cell duoi chay ablation DAY DU: 8 bien the x 5 fold (`--all_folds`) -- tuc **toi da 40 luot train rieng biet**, moi luot toi da `FULL_EPOCHS` epoch. Day la nguyen nhan chinh khien phien chay bi huy ("outtime"): du moi epoch nhanh, cong don 40 luot train se ton rat nhieu gio -- KHONG phai do 40 hay 100 epoch tu than no cham.

**2 nguyen nhan pho bien khien phien bi huy tren Kaggle:**
1. **Vuot gioi han session** (GPU toi da ~9 tieng lien tuc, cong quota GPU/tuan) -- xay ra khi tong thoi gian 40 luot train that su vuot qua.
2. **Idle-disconnect**: neu ban chi bam Run tung cell (Edit mode) roi dong tab / may sleep, Kaggle co the ngat phien du chua het compute -- rat de nham voi (1). **Cach tranh**: dung **Save Version -> Save & Run All (Commit)** (goc tren ben phai) thay vi Run tung cell -- notebook se chay nen (background), khong phu thuoc vao tab trinh duyet con mo hay khong.

**Neu van bi huy giua chung (du da dung Commit)**: KHONG can chay lai tu dau. Nho co `--resume` trong lenh o cell duoi, script se tu dong doc `*_fold{F}_partial.csv` da luu san trong `FULL_OUTPUT_DIR`, bo qua cac bien the/fold da xong, va chi chay tiep phan con thieu. Chi can Commit lai (chay lai chinh cell nay) nhieu lan cho den khi xong het.

**De giam rui ro out-of-time ngay tu dau**, cell duoi da giam `FULL_EPOCHS`/`FULL_PATIENCE` xuong muc thuc te hon (60 / 12) so voi ly tuong (100 / 20) -- ban co the tang lai neu thay con du thoi gian, hoac giam sau hon / bo `--all_folds` (chay tung `--fold` rieng qua nhieu lan Commit) neu van khong du.

**Ve GPU: dung T4 x2, KHONG dung P100.** Ly do ban dau chon P100 (bang thong bo nho cao hon, hop voi `torch.fft.fft` memory-bandwidth-bound) van dung ve mat ly thuyet, NHUNG thuc te tren Kaggle hien tai, ban PyTorch preinstalled da BAO LOI `CUDA error: no kernel image is available for execution on the device` tren P100 -- cac ban PyTorch gan day (>= 2.1) thuong da bo kernel cho kien truc Pascal (sm_60) cua P100. Cell kiem tra GPU/PyTorch ngay dau notebook (sau `nvidia-smi`) se tu dong phat hien va bao loi ro rang neu ban accelerator dang chon khong tuong thich.

In [ ]:
FULL_EPOCHS = 60      # giam tu 100 -> 60 de giam rui ro out-of-time; tang lai neu con du gio
FULL_PATIENCE = 12    # giam tu 20 -> 12 tuong ung (early-stop som hon mot chut)
FULL_BATCH_SIZE = 256
FULL_OUTPUT_DIR = '/kaggle/working/ablation_full_results'

os.makedirs(FULL_OUTPUT_DIR, exist_ok=True)

# --resume: neu lan chay truoc bi Kaggle huy giua chung, chay lai CHINH XAC lenh nay
# (Commit lai) se tu dong bo qua bien the/fold da xong, khong mat cong da train.
!python code/ablation_fft.py \
    --dataset {DATASET} --running_set {RUNNING_SET} \
    --epochs {FULL_EPOCHS} --patience {FULL_PATIENCE} --batch_size {FULL_BATCH_SIZE} \
    --all_folds \
    --cuda 0 \
    --output_dir {FULL_OUTPUT_DIR} \
    --resume

## Doc ket qua full ablation (mean +/- std qua 5 fold)

Cell duoi doc file `ablation_fnet_{dataset}_{running_set}_AGGREGATED_*.csv` (duoc `ablation_fft.py` tu dong tao khi chay voi `--all_folds`), chua ket qua trung binh +/- do lech chuan qua 5 fold cho tung bien the -- day la bang dung de dua vao report/paper.

In [ ]:
import glob
import pandas as pd

agg_files = sorted(glob.glob(os.path.join(FULL_OUTPUT_DIR, f'ablation_fnet_{DATASET}_{RUNNING_SET}_AGGREGATED_*.csv')))

if agg_files:
    agg_csv = agg_files[-1]
    print('Doc ket qua tong hop (DA HOAN TAT ca 5 fold) tu:', agg_csv)
    df_full = pd.read_csv(agg_csv)
else:
    # Chua co file AGGREGATED nghia la lan chay --all_folds chua di het ca 5 fold
    # (vi du bi Kaggle huy giua chung truoc khi chay xong). Fallback: tu gop cac
    # file *_fold{F}_partial.csv (checkpoint cua tung fold, luon duoc cap nhat sau
    # MOI bien the) de xem ket qua so bo cua nhung fold da xong toi thoi diem nay.
    partial_files = sorted(glob.glob(os.path.join(
        FULL_OUTPUT_DIR, f'ablation_fnet_{DATASET}_{RUNNING_SET}_fold*_partial.csv')))
    assert partial_files, ('Khong tim thay ca file AGGREGATED lan file *_partial.csv nao -- '
                            'kiem tra lai cell chay full o tren da bat dau chay chua.')
    print(f'Chua co file AGGREGATED (--all_folds chua chay xong het 5 fold). '
          f'Tam gop ket qua so bo tu {len(partial_files)} file checkpoint:')
    for p in partial_files:
        print(' -', p)

    frames = [pd.read_csv(p) for p in partial_files]
    df_all = pd.concat(frames, ignore_index=True)
    df_all = df_all[df_all['status'] == 'success']

    df_full = (
        df_all.groupby('variant')
        .agg(description=('description', 'first'),
             ci_mean=('ci', 'mean'), ci_std=('ci', 'std'),
             mse_mean=('mse', 'mean'), mse_std=('mse', 'std'),
             r2_mean=('r2', 'mean'), r2_std=('r2', 'std'),
             n_folds=('ci', 'count'))
        .reset_index()
    )
    print('\nLuu y: n_folds < 5 nghia la bien the do chua chay du 5 fold, so mean/std chi la so bo.')

df_full_sorted = df_full.sort_values('ci_mean', ascending=False).reset_index(drop=True)
show_cols = [c for c in ['variant', 'description', 'ci_mean', 'ci_std', 'mse_mean', 'mse_std',
                          'r2_mean', 'r2_std', 'n_folds'] if c in df_full_sorted.columns]
df_full_sorted[show_cols]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4.5))
plt.bar(df_full_sorted['variant'], df_full_sorted['ci_mean'], yerr=df_full_sorted['ci_std'],
        capsize=4, color='#4C72B0')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Test CI (mean +/- std, 5-fold)')
plt.title(f'FFT Component Ablation (FULL, {FULL_EPOCHS} epoch, 5-fold) - {DATASET}-{RUNNING_SET}')
plt.tight_layout()
plt.savefig(os.path.join(FULL_OUTPUT_DIR, 'full_ci_comparison.png'), dpi=150)
plt.show()

## Hoan tat

Ket qua day du nam trong `FULL_OUTPUT_DIR` (`/kaggle/working/ablation_full_results`):
- `ablation_fnet_*_fold{{0..4}}_*.csv/.json` -- ket qua chi tiet tung fold, tung bien the
- `ablation_fnet_*_AGGREGATED_*.csv` -- bang mean +/- std dung de bao cao
- `full_ci_comparison.png` -- bieu do so sanh CI (co error bar) giua 8 bien the

Vao tab **Output** cua Kaggle de tai toan bo cac file nay ve may.